In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from prophet.plot import add_changepoints_to_plot
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import os

# =========================================================
# 1. CẤU HÌNH & TẢI DỮ LIỆU
# =========================================================
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
DATA_PATH = "processed/KPI_requirements/kpi_daily_2021.csv"
FULL_PATH = os.path.join(BASE_DIR, DATA_PATH)

# Fallback đường dẫn tuyệt đối  (dự phòng)
if not os.path.exists(FULL_PATH):
    FULL_PATH = "E:/BTL/2526-LTXLDL-Project-Taxi2021-group1.3/Data/processed/KPI_requirements/kpi_daily_2021.csv"

print(f"📂 Đang đọc dữ liệu từ: {FULL_PATH}")
if not os.path.exists(FULL_PATH):
    print("❌ LỖI: Không tìm thấy file dữ liệu.")
    exit()

df = pd.read_csv(FULL_PATH)

# =========================================================
# 2. TIỀN XỬ LÝ DỮ LIỆU (PREPROCESSING)
# =========================================================
df_model = df[['pickup_date', 'trips']].rename(columns={'pickup_date': 'ds', 'trips': 'y'})
df_model['ds'] = pd.to_datetime(df_model['ds'])
df_model = df_model.sort_values('ds').reset_index(drop=True)

# Chia tập Train (Học) và Test (Kiểm tra)
TEST_DAYS = 14
train = df_model.iloc[:-TEST_DAYS]
test = df_model.iloc[-TEST_DAYS:]

print(f"📊 Tổng số ngày dữ liệu: {len(df_model)}")
print(f"   - Train: {len(train)} ngày (Kết thúc: {train['ds'].max().date()})")
print(f"   - Test:  {len(test)} ngày (Bắt đầu: {test['ds'].min().date()})")

# =========================================================
# 3. HUẤN LUYỆN MÔ HÌNH (CẤU HÌNH CHO DATA COVID/PHỤC HỒI)
# =========================================================
# Tinh chỉnh tham số quan trọng cho năm 2021 (Năm phục hồi sau đại dịch):
# 1. seasonality_mode='multiplicative': Vì biên độ dao động (cuối tuần vs ngày thường) 
#    tăng lên khi tổng lượng khách hồi phục.
# 2. changepoint_prior_scale=0.15 (Mặc định 0.05): Tăng độ linh hoạt để mô hình 
#    bắt kịp các đợt mở cửa/phong tỏa hoặc sự kiện lớn thay đổi xu hướng.

# Tinh chỉnh lại tham số để chống overfitting khi thiếu dữ liệu chu kỳ năm
model = Prophet(
    yearly_seasonality=False, # Tắt đi vì có chưa đủ 2 năm, bật lên chỉ làm nhiễu mô hình
    weekly_seasonality=True, 
    daily_seasonality=False,
    seasonality_mode='additive', # Chuyển về additive để kiểm soát biên độ an toàn hơn
    changepoint_prior_scale=0.01 # Hạ thấp xuống (mặc định 0.05) để cố định đường xu hướng phẳng
)

# Thêm ngày lễ công cộng
model.add_country_holidays(country_name='US')

# Tạo chuỗi ngày nghỉ Giáng Sinh kéo dài tại NYC
christmas_weeks = pd.DataFrame({
  'holiday': 'christmas_shutdown',
  'ds': pd.to_datetime([f'2021-12-{d}' for d in range(23, 32)]),
  'lower_window': 0,
  'upper_window': 0,
})

# Khai báo vào mô hình
model = Prophet(
    weekly_seasonality=True,
    seasonality_mode='additive',
    changepoint_prior_scale=0.01,
    holidays=christmas_weeks # Nạp chuỗi ngày nghỉ tự định nghĩa vào đây
)

print("🚀 Đang huấn luyện mô hình Prophet (Chế độ thích ứng biến động)...")
model.fit(train)

# =========================================================
# 4. ĐÁNH GIÁ MÔ HÌNH (EVALUATION)
# =========================================================
future_test = model.make_future_dataframe(periods=TEST_DAYS)
forecast = model.predict(future_test)

# Lấy dữ liệu 14 ngày cuối để so sánh
forecast_test = forecast.iloc[-TEST_DAYS:]
y_true = test['y'].values
y_pred = forecast_test['yhat'].values

mae = mean_absolute_error(y_true, y_pred)
mape = mean_absolute_percentage_error(y_true, y_pred)

print("\n" + "="*40)
print("🔍 KẾT QUẢ ĐÁNH GIÁ (TRÊN TẬP TEST 14 NGÀY)")
print("="*40)
print(f"MAE  (Sai số tuyệt đối): {mae:,.0f} chuyến/ngày")
print(f"MAPE (Sai số phần trăm): {mape*100:.2f}%")

if mape < 0.1:
    print("🌟 Đánh giá: Mô hình RẤT TỐT (<10%)")
elif mape < 0.2:
    print("✅ Đánh giá: Mô hình TỐT (<20%)")
else:
    print("⚠️ Đánh giá: Cần cải thiện tham số (<30%)")

# =========================================================
# 5. DỰ BÁO TƯƠNG LAI & XUẤT KẾT QUẢ
# =========================================================
print("\n🔮 Đang dự báo cho 14 ngày tiếp theo (Chưa có dữ liệu)...")
# Tạo khung thời gian cho 14 ngày SAU KHI hết dữ liệu trong file
last_date = df_model['ds'].max()
future_next = model.make_future_dataframe(periods=TEST_DAYS + 14) 
# Lọc lấy đúng phần tương lai mới
future_days_only = future_next[future_next['ds'] > last_date].copy()

forecast_new = model.predict(future_days_only)

print("\n--- DỰ BÁO NHU CẦU 5 NGÀY TỚI ---")
print(forecast_new[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].head(5))

# =========================================================
# 6. TRỰC QUAN HÓA CHUYÊN SÂU (VISUALIZATION)
# =========================================================
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# --- BIỂU ĐỒ 1: SO KHỚP THỰC TẾ VS DỰ BÁO ---
plt.figure(figsize=(14, 7))
plt.plot(train['ds'], train['y'], label='Dữ liệu huấn luyện', color='#1f77b4', alpha=0.6)
plt.plot(test['ds'], test['y'], label='Thực tế (Test)', color='green', linewidth=2)
plt.plot(forecast_test['ds'], forecast_test['yhat'], label='Dự báo (Test)', color='red', linestyle='--', linewidth=2)
plt.title('Dự báo Nhu cầu Taxi NYC 2021 (Kiểm thử mô hình)', fontsize=16)
plt.xlabel('Thời gian')
plt.ylabel('Số lượng chuyến đi')
plt.legend()    
plt.grid(True, linestyle=':', alpha=0.6)
plt.savefig(os.path.join(FIGURES_DIR, 'forecast_evaluation.png'), dpi=150)
print(f"\n✅ Đã lưu biểu đồ đánh giá tại folder {FIGURES_DIR}")
plt.show()

# --- BIỂU ĐỒ 2: PHÂN TÍCH ĐIỂM GÃY XU HƯỚNG (CHANGE POINTS) ---
# Biểu đồ này cực kỳ quan trọng để giải thích yếu tố COVID
fig_cp = model.plot(forecast)
a = add_changepoints_to_plot(fig_cp.gca(), model, forecast)
plt.title('Các điểm thay đổi xu hướng (Change Points) - Phản ánh biến động thị trường', fontsize=14)
plt.savefig(os.path.join(FIGURES_DIR, 'forecast_changepoints.png'), dpi=150)
plt.show()

# --- BIỂU ĐỒ 3: PHÂN RÃ CÁC THÀNH PHẦN ---
fig_comp = model.plot_components(forecast)
plt.savefig(os.path.join(FIGURES_DIR, 'forecast_components.png'), dpi=150)
plt.show()

print("\n🎉 HOÀN TẤT! Hãy kiểm tra các biểu đồ để đưa vào báo cáo.")